In [1]:
import torch
from icecream import ic

In [2]:
device = torch.device("cuda")

In [3]:
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to(device)
histogram_shape = (3, 4)
positions = torch.rand(100, 100_000, 2).to(device)
bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)

In [4]:
%%timeit
# Normalise particle coordinates to normalised bin space
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

getattr(torch, device.type).synchronize()

717 μs ± 2.23 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
# I have seen 500 us on MPS here (where in the same kernel instance the copy of this
# code block below took 16 ms, which this no seems to need on every restart ... weird)

# Similar effect on CUDA with 50 us and 700 us

In [6]:
del extent, histogram_shape, positions, bin_widths

In [7]:
positions = (torch.rand(100, 100_000).to(device), torch.rand(100, 100_000).to(device))
bins = (
    torch.linspace(0.0, 1.0, 500).to(device),
    torch.linspace(0.0, 1.0, 500).to(device),
)
grid_sizes = []
spacings = []
for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)

In [8]:
%%timeit
# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []
for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)

getattr(torch, device.type).synchronize()

740 μs ± 396 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
del positions, bins, grid_sizes, spacings, i, bin_array, spacing, diffs

In [10]:
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to(device)
histogram_shape = (3, 4)
positions = torch.rand(100, 100_000, 2).to(device)
bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)

In [11]:
%%timeit
# Normalise particle coordinates to normalised bin space
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

getattr(torch, device.type).synchronize()

716 μs ± 643 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
del extent, histogram_shape, positions, bin_widths